# ML Potential with ACSF Features
In this lab, we will learn a potential energy surface (PES) with a feedforward neural network (FNN).

We use the ACSF features to represent the molecule.

Python Libraries:

1. `torch_geometric`: used to download simple MD trajectories.
2. `ase`: a wide used atomic simulation environment library to handle molecules [Website](https://ase-lib.org/).
3. `dscribe`: a Python package for transforming atomic structures into fixed-size numerical features. [Website](https://singroup.github.io/dscribe/latest/). You could also write your own ACSF function!
4. `torch`: for FNN training.
5. `numpy`: matrix arithmetics.
6. `random`: to randomly sample frames.

Optional for visualization:
`py3Dmol` and `Ipython.display`.

## Dataset
The dataset we will use is MD17. Since this is a widely used dataset for ML potential, PyTorch Geometric (PyG) provides direct downloading. You can look at the available molecules [here](https://pytorch-geometric.readthedocs.io/en/2.5.0/generated/torch_geometric.datasets.MD17.html).

In [1]:
! pip install -q torch-geometric dscribe py3Dmol


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 777.4/777.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 10.4 MB/s eta 0:00:00


In [2]:
from torch_geometric.datasets import MD17
from ase import Atoms
import torch
import torch.nn as nn
import numpy as np
from dscribe.descriptors import ACSF
import random

## Download MD17 dataset
We will use the ethanol molecule for today's demonstration.

In [3]:
dataset = MD17(root="lab9/MD17", name='ethanol') # root is where we save the downloaded data, name is the name used to querry the MD trajectories.
n_data = len(dataset)
print("Number of MD frames for ethanol:", n_data)

Processing...


Number of MD frames for ethanol: 555092


Done!


### Visualize the trajectory
We plot the 3D shapes of the ethanol molecule at every 1000 snapshots of the dataset.

In [4]:
import py3Dmol
from IPython.display import display, HTML

In [5]:
def atoms_to_xyz(atoms): # turn ASE atoms into xyz
    xyz = f"{len(atoms)}\n\n"
    for a in atoms:
        a_sym = a.symbol
        x,y,z = a.position
        xyz += f"{a_sym}    {x}     {y}     {z}\n"
    return xyz

def show_one_frame(atoms, width=300, height=300, scale=1.0):
    xyz = atoms_to_xyz(atoms)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(xyz, "xyz")
    view.setStyle({'stick': {}, 'sphere':{"scale": 0.3}})
    view.zoomTo()
    view.show()

def plot_multiple_frames(dataset, n_frames=12, step=100):
    '''
    plot many frames
    '''
    width, height = 250, 250

    # Create HTML container with viewers side by side
    html = "<div style='display:flex; flex-wrap:wrap;'>"

    for i in range(n_frames):
        p = i*step
        numbers = dataset[p].z.numpy()
        pos = dataset[p].pos.numpy()
        atoms = Atoms(numbers=numbers, positions=pos) # a molecule
        xyz_str = atoms_to_xyz(atoms)
        view = py3Dmol.view(width=width, height=height)
        view.addModel(xyz_str, 'xyz')
        view.setStyle({'stick': {}})
        view.zoomTo()
        html += f"<div style='margin:1px'>{view._make_html()}</div>"

    html += "</div>"

    display(HTML(html))

In [6]:
plot_multiple_frames(dataset, n_frames=12, step=1000)

## Now we start to train the FNN

### First, we randomly sample 500 frames from the dataset to train, so we won't sit here all evening!

In [7]:
n_sample = 500
indices = random.sample(range(n_data), n_sample) # no duplicate
frames = [dataset[i] for i in indices]

#### Next, we extract the $(\mathbf{R}_i, E, \mathbf{F}_i)$ values, although we won't use the atomic forces today.

In [8]:
frame_list = [] # xyz information
energies = [] # total energy
forces_list = [] # atomic force
for frame in frames:
    atomic_numbers = frame.z.numpy() # list of atomic numbers of the atoms in the molecule
    pos = frame.pos.numpy() # 2D array, each row is the xyz coordinate of the corresponding atom
    energy = frame.energy.item() # float, total energy
    force = frame.force.numpy() #2D array, each row is the force applied onto the atom
    atoms = Atoms(numbers=atomic_numbers, positions=pos) # an ASE atom object
    frame_list.append(atoms)
    energies.append(energy)
    forces_list.append(force)

energies = np.array(energies) # 1D array of (n_sample,)
forces_list = np.array(forces_list) # 3D array (n_sample, n_atom, 3)

In [9]:
# let's have a look:
print(atomic_numbers)
print(pos)
print(energy)
print(force)

[6 6 8 1 1 1 1 1 1]
[[-0.10613168 -0.1814316  -0.5031363 ]
 [-0.3687598   1.3208947  -0.15280429]
 [ 0.3918999  -1.0307815   0.60537285]
 [-1.1439807  -0.64524925 -0.6206492 ]
 [ 0.48050845 -0.2199548  -1.488433  ]
 [-0.02937278  2.152675   -0.8937748 ]
 [-1.4358568   1.5369344   0.04571408]
 [ 0.30636686  1.4498593   0.6432404 ]
 [ 1.2594237  -1.4917351   0.5208425 ]]
-97191.0390625
[[-27.157251  -18.890278  -25.73558  ]
 [  6.7421546   8.753801  -39.282616 ]
 [  3.2907119  17.618153   -2.2614732]
 [ 26.932644    9.468257  -13.107433 ]
 [-11.158189  -11.037629   29.792358 ]
 [ -9.242427  -38.83132    11.483639 ]
 [ 11.258478   -3.9536738  10.325802 ]
 [ 11.072934   17.410357   44.391174 ]
 [-11.603276   19.45249   -15.618567 ]]


### Next, we generate the ACSF features

In [10]:
species = ["H", "C", "O"]

acsf = ACSF(
    species=species,
    r_cut=6.0,
    g2_params=[[1, 0.0], [1, 1.0], [2.0, 1.0]], # we choose 3 sets of parameters for G2
    g4_params=[[1, 1, 1], [1, 2, 1], [1, 1, -1]], # 3 sets of parameters for G4
)


In [11]:
X = []
for atoms in frame_list:
    X.append(acsf.create(atoms)) # greate matrices for each molecule

X = np.array(X)  # shape: (n_frames, n_atoms, n_features)
print("Shape of the feature matrix: ", X.shape)
n_feature = X.shape[-1] # the number of features per atom

Shape of the feature matrix:  (500, 9, 30)


## Construct the FNN model
We will use the `tanh` activation since the atomic energy can be negative.

In [12]:
class AtomicNN(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 50), # n_in is the feature vector length per atom.
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 1)
        )
    def forward(self, x):
        return self.net(x)

### Remember that for each species, we train a separate model

In [13]:
unique_Z = [1, 6, 8] # H, C, O
models = {Z: AtomicNN(n_feature) for Z in unique_Z} # a dictionary of models

### Initialize the optimizer
Although we have three models to train, we can put all parameters together and optimize them.

In [14]:
all_params = []
for Z, model in models.items():
    for p in model.parameters():
        all_params.append(p)

optimizer = torch.optim.Adam(
    all_params,
    lr=1e-1
)

### Train-test split

In [15]:
# train-test split
# Since we randomly sampled the frames, we can simply use the last 20% as the test set
n_train = int(n_sample * 0.8)
X_train, X_test = X[:n_train], X[n_train:]
energies_train, energies_test = energies[:n_train], energies[n_train:]
frames_train,  frames_test= frame_list[:n_train], frame_list[n_train:]

### Wrap the energy evaluation into a function

In [16]:
def total_energy(X_frame, Z_frame):
    '''
    X is the feature matrix, Z is the list of atomic numbers for all atoms in the molecule.
    '''
    E = 0.0
    i = 0
    for Z in Z_frame:
        x = torch.tensor(X_frame[i], dtype=torch.float32)
        E += models[Z](x)
        i += 1
    return E

### Now Let's start training!

In [17]:
batch_size = 10 # train 10 frames per batch. if you use a small batch, make sure you reduce the learning rate value!

loss_old = 1e10 # the last step loss to flag improvement

for epoch in range(500):
    loss_epoch = 0.0
    # process batches manually
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]
        E_batch = energies_train[i:i+batch_size]
        atoms_batch = frames_train[i:i+batch_size]

        optimizer.zero_grad()
        batch_loss = 0.0
        for X_atom, E_ref, atoms in zip(X_batch, E_batch, atoms_batch):
            # optimizer.zero_grad()
            E_pred = total_energy(X_atom, atoms.numbers)
            loss = (E_pred - torch.tensor(E_ref, dtype=torch.float32))**2
            batch_loss += loss
        batch_loss.backward()
        optimizer.step()
        loss_epoch += batch_loss.item()
    loss_epoch /= n_train
    if epoch > 20 and (loss_old - loss_epoch < 10):
        print("Loss not improving, stop.")
        print(f"Epoch {epoch+1:3d} | Loss {loss_epoch:.2f}")
        break
    loss_old = loss_epoch
    if epoch==0 or ((epoch+1) % 10 == 0):
        print(f"Epoch {epoch+1:3d} | Loss {loss_epoch:.2f}")



Epoch   1 | Loss 9284170588.16
Epoch  10 | Loss 6473692774.40
Epoch  20 | Loss 4157401272.32
Epoch  30 | Loss 2520557701.12
Epoch  40 | Loss 1419338524.16
Epoch  50 | Loss 726618488.32
Epoch  60 | Loss 328877878.40
Epoch  70 | Loss 126968161.28
Epoch  80 | Loss 39975581.04
Epoch  90 | Loss 9719475.58
Epoch 100 | Loss 1709946.75
Epoch 110 | Loss 201503.17
Epoch 120 | Loss 14481.75
Epoch 130 | Loss 581.60
Loss not improving, stop.
Epoch 139 | Loss 35.10


## Evaluate the model with the test set

In [18]:
with torch.no_grad():
    E_pred_test = torch.tensor([total_energy(Xb, a.numbers) for Xb, a in zip(X_test, frames_test)])
    E_ref_test  = torch.tensor(energies_test, dtype=torch.float32)

    test_mse = ((E_pred_test - E_ref_test)**2).mean().item()
    test_mae = (E_pred_test - E_ref_test).abs().mean().item()

print(f"Test MSE: {test_mse:.4f}")
print(f"Test MAE: {test_mae:.4f}")


Test MSE: 30.3222
Test MAE: 4.6595


## Save the model
You can directly use torch to save trained models.

In [19]:
for Z, model in models.items():
    torch.save(model.state_dict(), f"model_atom_{Z}.pt") # .pt is a pytorch object

In [20]:
# load model
# Recreate the model architecture
n_features = acsf.get_number_of_features()
models = {}
for Z in unique_Z:
  model = AtomicNN(n_features)
  # Load weights
  model.load_state_dict(torch.load(f"model_atom_{Z}.pt"))
  # model.eval()  # set to evaluation mode
  models[Z] = model

{1: AtomicNN(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=50, bias=True)
    (1): Tanh()
    (2): Linear(in_features=50, out_features=50, bias=True)
    (3): Tanh()
    (4): Linear(in_features=50, out_features=1, bias=True)
  )
), 6: AtomicNN(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=50, bias=True)
    (1): Tanh()
    (2): Linear(in_features=50, out_features=50, bias=True)
    (3): Tanh()
    (4): Linear(in_features=50, out_features=1, bias=True)
  )
), 8: AtomicNN(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=50, bias=True)
    (1): Tanh()
    (2): Linear(in_features=50, out_features=50, bias=True)
    (3): Tanh()
    (4): Linear(in_features=50, out_features=1, bias=True)
  )
)}
